In [2]:
import pandas as pd
import glob
import os

base_folder = r"C:\Users\Huawei\Desktop\témalab1\Bubi-data-analyzis---T-malabor1\code\Adatok"
years = ["2022", "2023", "2024"]

dfs = []

# Regex a helyes formátumú helyazonosítókhoz
pattern = r'^\d{4}-'

for year in years:
    folder_path = os.path.join(base_folder, year)
    file_list = glob.glob(os.path.join(folder_path, "*.xlsx"))
    print(f"{year}: {len(file_list)} fájl található")

    for file in file_list:
        print("Beolvasás:", file)
        temp_df = pd.read_excel(file)
        temp_df["year"] = int(year)

        # Már itt szűrünk — csak azokat tartsuk meg, ahol a helyazonosítók jók
        mask = (
            temp_df['start_place_id'].astype(str).str.match(pattern) &
            temp_df['end_place_id'].astype(str).str.match(pattern)
        )
        temp_df = temp_df[mask]

        # Csak a releváns oszlopokat tartsuk meg (opcionális gyorsítás)
        cols = ['start_time', 'end_time', 'duration', 'start_place_id', 'end_place_id',
                'start_lat', 'start_lng', 'end_lat', 'end_lng',
                'cust_id', 'bike_id', 'coupon_name', 'year']
        temp_df = temp_df[cols]

        dfs.append(temp_df)

# Összefűzés
df = pd.concat(dfs, ignore_index=True)
print("Összes sor beolvasva és szűrve:", len(df))

# Dátumkonverzió
df['start_time'] = pd.to_datetime(df['start_time'], errors='coerce')
df['end_time'] = pd.to_datetime(df['end_time'], errors='coerce')
df = df.dropna(subset=['start_time', 'end_time'])

# Időtartam percben
df['duration_min'] = (df['end_time'] - df['start_time']).dt.total_seconds() / 60
df = df[(df['duration_min'] > 0) & (df['duration_min'] < 120)]

2022: 12 fájl található
Beolvasás: C:\Users\Huawei\Desktop\témalab1\Bubi-data-analyzis---T-malabor1\code\Adatok\2022\2022_01.xlsx
Beolvasás: C:\Users\Huawei\Desktop\témalab1\Bubi-data-analyzis---T-malabor1\code\Adatok\2022\2022_02.xlsx
Beolvasás: C:\Users\Huawei\Desktop\témalab1\Bubi-data-analyzis---T-malabor1\code\Adatok\2022\2022_03.xlsx
Beolvasás: C:\Users\Huawei\Desktop\témalab1\Bubi-data-analyzis---T-malabor1\code\Adatok\2022\2022_04.xlsx
Beolvasás: C:\Users\Huawei\Desktop\témalab1\Bubi-data-analyzis---T-malabor1\code\Adatok\2022\2022_05.xlsx
Beolvasás: C:\Users\Huawei\Desktop\témalab1\Bubi-data-analyzis---T-malabor1\code\Adatok\2022\2022_06.xlsx
Beolvasás: C:\Users\Huawei\Desktop\témalab1\Bubi-data-analyzis---T-malabor1\code\Adatok\2022\2022_07.xlsx
Beolvasás: C:\Users\Huawei\Desktop\témalab1\Bubi-data-analyzis---T-malabor1\code\Adatok\2022\2022_08.xlsx
Beolvasás: C:\Users\Huawei\Desktop\témalab1\Bubi-data-analyzis---T-malabor1\code\Adatok\2022\2022_09.xlsx
Beolvasás: C:\Users\Hu

In [3]:
df.to_pickle("bubi_all_years.pkl")

In [4]:
df = pd.read_pickle("bubi_all_years.pkl")

In [4]:
intros = {
    "csonka": pd.Timestamp("2022-06-15"),
    "zuglo": pd.Timestamp("2023-06-28"),
    "ferencvaros": pd.Timestamp("2023-09-01"),
    "kopaszi": pd.Timestamp("2024-04-09"),
    "westend": pd.Timestamp("2024-05-01")
}

In [5]:
areas = {
    "csonka": ["Csonka"],
    "zuglo": ["Zugló", "Sportaréna"],
    "ferencvaros": ["Haller", "Mester"],
    "kopaszi": ["Kopaszi"],
    "westend": ["Westend", "Balzac"]
}

In [15]:
import pandas as pd

# --- 1. Adat beolvasása ---
df = pd.read_pickle("bubi_all_years.pkl")

# --- 2. Állomáscsaládok és bevezetési dátumok ---
families = {
    "Csonka János tér (2022-06-15)": {
        "names": ["Csonka János tér"],
        "start": "2022-06-15"
    },
    "Zuglói csoport (2023-06-28)": {
        "names": [
             "Zugló vasútállomás", "Sportaréna"
        ],
        "start": "2023-06-28"
    },
    "9. kerület (2023-09-01)": {
        "names": [
            "Mester utca", "Haller utca"
        ],
        "start": "2023-09-01"
    },
    "Kopaszi-gát (2024-04-09)": {
        "names": ["Kopaszi"],
        "start": "2024-04-09"
    },
    "Westend – Balzac utca (2024-05-01)": {
        "names": ["Westend", "Balzac"],
        "start": "2024-05-01"
    }
}

# --- 3. Időkonverzió ---
df['start_time'] = pd.to_datetime(df['start_time'], errors='coerce')
df['end_time'] = pd.to_datetime(df['end_time'], errors='coerce')

# --- 4. Elemzés minden családra ---
for family_name, data in families.items():
    start_date = pd.to_datetime(data['start'])
    end_date = start_date + pd.Timedelta(days=90)
    names = data['names']

    print(f"\n=== {family_name} – {start_date.date()} utáni 90 nap ===")

    # Részleges illesztés mintája
    pattern = "|".join(names)

    # Csak a bevezetés utáni 90 nap adatai
    df_period = df[(df['start_time'] >= start_date) & (df['start_time'] < end_date)]

    # --- Kiáramlás: az adott csoportból indult utazások ---
    mask_out = df_period['start_place_id'].str.contains(pattern, case=False, na=False)
    outgoing = (
        df_period[mask_out]
        .groupby('end_place_id')
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(10)
    )

    # --- Beáramlás: az adott csoportba érkező utazások ---
    mask_in = df_period['end_place_id'].str.contains(pattern, case=False, na=False)
    incoming = (
        df_period[mask_in]
        .groupby('start_place_id')
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(10)
    )

    # --- Eredmények kiírása ---
    print("\nTOP 10 beáramló állomás (honnan jöttek):")
    print(incoming if not incoming.empty else "Nincs adat erre az időszakra.")

    print("\nTOP 10 kiáramló állomás (hova mentek):")
    print(outgoing if not outgoing.empty else "Nincs adat erre az időszakra.")



=== Csonka János tér (2022-06-15) – 2022-06-15 utáni 90 nap ===

TOP 10 beáramló állomás (honnan jöttek):
                                start_place_id  count
106                1107-Móricz Zsigmond körtér    191
127                      1128-Csonka János tér    176
130  1131-Kelenföld vasútállomás M (Etele tér)    136
119                         1120-Budapart Gate    127
108                      1109-Újbuda-központ M    104
110                 1111-Kosztolányi Dezső tér     78
129  1130-Hauszmann Alajos utca - Fehérvári út     68
124                          1125-Bikás park M     65
122              1123-Karolina út - Tétényi út     58
118                   1119-Infopark - aluljáró     53

TOP 10 kiáramló állomás (hova mentek):
                                  end_place_id  count
105                1107-Móricz Zsigmond körtér    179
126                      1128-Csonka János tér    176
118                         1120-Budapart Gate    163
129  1131-Kelenföld vasútállomás M (Etele t

In [16]:
import pandas as pd
import networkx as nx
import folium
import os

# --- 1. Adat beolvasása ---
df = pd.read_pickle("bubi_all_years.pkl")

# --- 2. Állomáscsaládok és bevezetési dátumok ---
families = {
    "Csonka János tér (2022-06-15)": {
        "names": ["Csonka János tér"],
        "start": "2022-06-15"
    },
    "Zuglói csoport (2023-06-28)": {
        "names": [
            "Egressy út", "Stefánia", "Zugló vasútállomás",
            "Kacsóh Pongrác", "Sportaréna", "Reiner Frigyes"
        ],
        "start": "2023-06-28"
    },
    "9. kerület (2023-09-01)": {
        "names": [
            "Mester utca", "Haller utca", "Nádasdy utca", "Soroksári út"
        ],
        "start": "2023-09-01"
    },
    "Kopaszi-gát (2024-04-09)": {
        "names": ["Kopaszi"],
        "start": "2024-04-09"
    },
    "Westend – Balzac utca (2024-05-01)": {
        "names": ["Westend", "Balzac"],
        "start": "2024-05-01"
    }
}

# --- 3. Dátumkonverzió ---
df['start_time'] = pd.to_datetime(df['start_time'], errors='coerce')
df['end_time'] = pd.to_datetime(df['end_time'], errors='coerce')

# --- 4. Mentési mappa ---
output_folder = "bubi_maps"
os.makedirs(output_folder, exist_ok=True)

# --- 5. Minden család feldolgozása ---
for family_name, data in families.items():
    start_date = pd.to_datetime(data['start'])
    end_date = start_date + pd.Timedelta(days=90)
    names = data['names']
    pattern = "|".join(names)

    print(f"\n=== {family_name} – {start_date.date()} utáni 90 nap ===")

    # Csak a bevezetés utáni 90 nap adatai
    df_period = df[(df['start_time'] >= start_date) & (df['start_time'] < end_date)]

    # --- Kiáramlás ---
    mask_out = df_period['start_place_id'].str.contains(pattern, case=False, na=False)
    outgoing = (
        df_period[mask_out]
        .groupby(['start_place_id', 'end_place_id', 'end_lat', 'end_lng'])
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(10)
    )

    # --- Beáramlás ---
    mask_in = df_period['end_place_id'].str.contains(pattern, case=False, na=False)
    incoming = (
        df_period[mask_in]
        .groupby(['start_place_id', 'start_lat', 'start_lng', 'end_place_id'])
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(10)
    )

    # Ha nincs adat, ugorjunk
    if outgoing.empty and incoming.empty:
        print("Nincs adat erre az időszakra.")
        continue

    # --- Gráf létrehozása ---
    G = nx.DiGraph()

    # Kiáramló élek (piros)
    for _, row in outgoing.iterrows():
        G.add_edge(row['start_place_id'], row['end_place_id'], weight=row['count'], direction="out")

    # Beáramló élek (zöld)
    for _, row in incoming.iterrows():
        G.add_edge(row['start_place_id'], row['end_place_id'], weight=row['count'], direction="in")

    # --- Koordináták hozzárendelése ---
    coords = {}
    for _, r in df_period.iterrows():
        if pd.notna(r['start_lat']) and pd.notna(r['start_lng']):
            coords[r['start_place_id']] = (r['start_lat'], r['start_lng'])
        if pd.notna(r['end_lat']) and pd.notna(r['end_lng']):
            coords[r['end_place_id']] = (r['end_lat'], r['end_lng'])

    # --- Térkép ---
    m = folium.Map(location=[47.4979, 19.0402], zoom_start=12, tiles="CartoDB positron")

    # Család állomásai (kék)
    for node, (lat, lng) in coords.items():
        color = "blue" if any(n in node for n in names) else "gray"
        folium.CircleMarker(
            location=[lat, lng],
            radius=5 if color == "blue" else 3,
            color=color,
            fill=True,
            fill_opacity=0.8,
            popup=node
        ).add_to(m)

    # Élek rajzolása
    for u, v, d in G.edges(data=True):
        if u in coords and v in coords:
            color = "red" if d['direction'] == "out" else "green"
            folium.PolyLine(
                locations=[coords[u], coords[v]],
                color=color,
                weight=max(1, d['weight'] / 30),  # skálázás
                opacity=0.6
            ).add_to(m)

    # --- Mentés ---
    safe_name = family_name.replace(" ", "_").replace("–", "-").replace("(", "").replace(")", "")
    output_path = os.path.join(output_folder, f"{safe_name}.html")
    m.save(output_path)
    print(f"Térkép mentve: {output_path}")



=== Csonka János tér (2022-06-15) – 2022-06-15 utáni 90 nap ===
Térkép mentve: bubi_maps\Csonka_János_tér_2022-06-15.html

=== Zuglói csoport (2023-06-28) – 2023-06-28 utáni 90 nap ===
Térkép mentve: bubi_maps\Zuglói_csoport_2023-06-28.html

=== 9. kerület (2023-09-01) – 2023-09-01 utáni 90 nap ===
Térkép mentve: bubi_maps\9._kerület_2023-09-01.html

=== Kopaszi-gát (2024-04-09) – 2024-04-09 utáni 90 nap ===
Térkép mentve: bubi_maps\Kopaszi-gát_2024-04-09.html

=== Westend – Balzac utca (2024-05-01) – 2024-05-01 utáni 90 nap ===
Térkép mentve: bubi_maps\Westend_-_Balzac_utca_2024-05-01.html


In [6]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import contextily as ctx

# --- 1. Adatok beolvasása ---
df = pd.read_pickle("bubi_all_years.pkl")

# --- 2. Állomáscsaládok és dátumok ---
families = {
    "Csonka János tér (2022-06-15)": {
        "names": ["Csonka János tér"],
        "start": "2022-06-15"
    },
    "Zuglói csoport (2023-06-28)": {
        "names": [
            "Egressy út", "Stefánia", "Zugló vasútállomás",
            "Kacsóh Pongrác", "Sportaréna", "Reiner Frigyes"
        ],
        "start": "2023-06-28"
    },
    "9. kerület (2023-09-01)": {
        "names": [
            "Mester utca", "Haller utca", "Nádasdy utca", "Soroksári út"
        ],
        "start": "2023-09-01"
    },
    "Kopaszi-gát (2024-04-09)": {
        "names": ["Kopaszi"],
        "start": "2024-04-09"
    },
    "Westend – Balzac utca (2024-05-01)": {
        "names": ["Westend", "Balzac"],
        "start": "2024-05-01"
    }
}

# --- 3. Dátumkonverzió ---
df['start_time'] = pd.to_datetime(df['start_time'], errors='coerce')
df['end_time'] = pd.to_datetime(df['end_time'], errors='coerce')

# --- 4. Egyedi állomáslista (stabil pozíciókhoz) ---
stations = pd.concat([
    df[['start_place_id', 'start_lat', 'start_lng']].rename(
        columns={'start_place_id': 'place_id', 'start_lat': 'lat', 'start_lng': 'lng'}),
    df[['end_place_id', 'end_lat', 'end_lng']].rename(
        columns={'end_place_id': 'place_id', 'end_lat': 'lat', 'end_lng': 'lng'})
]).drop_duplicates(subset=['place_id']).reset_index(drop=True)

# --- 5. Elemzés minden családra ---
for family_name, data in families.items():
    start_date = pd.to_datetime(data['start'])
    end_date = start_date + pd.Timedelta(days=90)
    names = data['names']

    print(f"\n=== {family_name} – {start_date.date()} utáni 90 nap ===")

    pattern = "|".join(names)
    df_period = df[(df['start_time'] >= start_date) & (df['start_time'] < end_date)]

    # --- Beáramlás ---
    mask_in = df_period['end_place_id'].str.contains(pattern, case=False, na=False)
    incoming = (
        df_period[mask_in]
        .groupby(['start_place_id', 'end_place_id'])
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(10)
    )

    if incoming.empty:
        print("Nincs adat erre az időszakra.")
        continue

    # --- Gráf építés ---
    G = nx.DiGraph()
    for _, row in incoming.iterrows():
        G.add_edge(row['start_place_id'], row['end_place_id'], weight=row['count'])

    # --- Pozíciók a valós koordinátákból ---
    pos = {}
    for _, row in stations.iterrows():
        pos[row['place_id']] = (row['lng'], row['lat'])  # X=lng, Y=lat

    # --- Alap matplotlib setup ---
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_title(f"{family_name}\nTOP 10 beáramló kapcsolat ({start_date.date()}–{end_date.date()})", fontsize=11)

    # --- Nyilak vastagsága arányosan ---
    weights = [max(1, w / incoming['count'].max() * 5) for w in incoming['count']]

    # --- Csúcsok kirajzolása ---
    nx.draw_networkx_nodes(
        G, pos, node_size=30,
        node_color=['red' if any(n in node for n in names) else 'skyblue' for node in G.nodes()],
        ax=ax, alpha=0.9
    )

    # --- Élek rajzolása ---
    nx.draw_networkx_edges(
        G, pos, ax=ax,
        width=weights, edge_color='darkred',
        arrowstyle='-|>', arrowsize=12, alpha=0.8
    )

    # --- Címkék (csak a célállomásokra) ---
    labels = {n: n for n in G.nodes() if any(k in n for k in names)}
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=8, font_color='black', ax=ax)

    # --- Szám annotációk az élekre ---
    for u, v, d in G.edges(data=True):
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        xm, ym = (x1 + x2) / 2, (y1 + y2) / 2
        ax.text(xm, ym, str(d['weight']), fontsize=7, color='darkred', ha='center')

    # --- Háttértérkép (contextily) ---
    ax.set_aspect('equal')
    xmin, xmax = min(x for x, _ in pos.values()), max(x for x, _ in pos.values())
    ymin, ymax = min(y for _, y in pos.values()), max(y for _, y in pos.values())
    ax.set_xlim(xmin - 0.01, xmax + 0.01)
    ax.set_ylim(ymin - 0.01, ymax + 0.01)
    ctx.add_basemap(ax, crs='EPSG:4326', source=ctx.providers.CartoDB.Positron, alpha=0.6)

    ax.axis('off')

    # --- Kép mentése ---
    filename = f"bubi_incoming_{family_name.split('(')[0].strip().replace(' ', '_')}.png"
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()

    print(f"→ Mentve: {filename}")



=== Csonka János tér (2022-06-15) – 2022-06-15 utáni 90 nap ===
→ Mentve: bubi_incoming_Csonka_János_tér.png

=== Zuglói csoport (2023-06-28) – 2023-06-28 utáni 90 nap ===
→ Mentve: bubi_incoming_Zuglói_csoport.png

=== 9. kerület (2023-09-01) – 2023-09-01 utáni 90 nap ===
→ Mentve: bubi_incoming_9._kerület.png

=== Kopaszi-gát (2024-04-09) – 2024-04-09 utáni 90 nap ===
→ Mentve: bubi_incoming_Kopaszi-gát.png

=== Westend – Balzac utca (2024-05-01) – 2024-05-01 utáni 90 nap ===
→ Mentve: bubi_incoming_Westend_–_Balzac_utca.png


In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import contextily as ctx
from shapely.geometry import Point, LineString
import geopandas as gpd
import matplotlib.patheffects as path_effects

# --- 1. Adatok beolvasása ---
df = pd.read_pickle("bubi_all_years.pkl")

# --- 2. Célállomás-csoportok ---
families = {
    "Csonka János tér (2022-06-15)": {"names": ["Csonka János tér"], "start": "2022-06-15"},
    "Zugló – Sportaréna (2023-06-28)": {"names": ["Sportaréna"], "start": "2023-06-28"},
    "Zugló – Vasútállomás (2023-06-28)": {"names": ["Zugló vasútállomás"], "start": "2023-06-28"},
    "9. kerület – Haller–Mester (2023-09-01)": {"names": ["Haller utca", "Mester utca"], "start": "2023-09-01"},
    "Kopaszi-gát (2024-04-09)": {"names": ["Kopaszi"], "start": "2024-04-09"},
    "Westend – Balzac utca (2024-05-01)": {"names": ["Westend", "Balzac"], "start": "2024-05-01"}
}

# --- 3. Időkonverzió ---
df['start_time'] = pd.to_datetime(df['start_time'], errors='coerce')
df['end_time'] = pd.to_datetime(df['end_time'], errors='coerce')

# --- 4. Állomás-koordináták ---
def get_station_coords(df):
    start_coords = df.groupby('start_place_id')[['start_lat', 'start_lng']].mean().rename(
        columns={'start_lat': 'lat', 'start_lng': 'lng'})
    end_coords = df.groupby('end_place_id')[['end_lat', 'end_lng']].mean().rename(
        columns={'end_lat': 'lat', 'end_lng': 'lng'})
    all_coords = pd.concat([start_coords, end_coords]).groupby(level=0).mean()
    return all_coords

station_coords = get_station_coords(df)

# --- 5. Gráf készítés és plot ---
for family_name, data in families.items():
    start_date = pd.to_datetime(data['start'])
    end_date = start_date + pd.Timedelta(days=90)
    names = data['names']
    pattern = "|".join(names)

    # Időszűrés
    df_period = df[(df['start_time'] >= start_date) & (df['start_time'] < end_date)]

    # Be- és kiáramlás top10
    mask_in = df_period['end_place_id'].str.contains(pattern, case=False, na=False)
    mask_out = df_period['start_place_id'].str.contains(pattern, case=False, na=False)

    incoming = (df_period[mask_in]
                .groupby('start_place_id')
                .size()
                .reset_index(name='count')
                .sort_values('count', ascending=False)
                .head(10))
    
    outgoing = (df_period[mask_out]
                .groupby('end_place_id')
                .size()
                .reset_index(name='count')
                .sort_values('count', ascending=False)
                .head(10))

    if incoming.empty and outgoing.empty:
        print(f"{family_name}: nincs adat a 90 napos időszakban.")
        continue

    # --- Családközpont koordináta ---
    cluster_stations = [s for s in station_coords.index if any(n.lower() in s.lower() for n in names)]
    cluster_lat = station_coords.loc[cluster_stations, 'lat'].mean()
    cluster_lng = station_coords.loc[cluster_stations, 'lng'].mean()
    station_coords.loc[family_name] = [cluster_lat, cluster_lng]

    # --- Élek építése ---
    edges = []
    for _, row in incoming.iterrows():
        src, tgt, c = row['start_place_id'], family_name, row['count']
        if src in station_coords.index:
            edges.append((src, tgt, c, "in"))
    for _, row in outgoing.iterrows():
        src, tgt, c = family_name, row['end_place_id'], row['count']
        if tgt in station_coords.index:
            edges.append((src, tgt, c, "out"))

    if not edges:
        print(f"{family_name}: nincs megjeleníthető él.")
        continue

    # --- GeoDataFrame ---
    edge_geoms = []
    for src, tgt, c, direction in edges:
        p1 = Point(station_coords.loc[src, ['lng', 'lat']])
        p2 = Point(station_coords.loc[tgt, ['lng', 'lat']])
        edge_geoms.append({
            "u": src, "v": tgt, "weight": c, "direction": direction,
            "geometry": LineString([p1, p2])
        })
    gdf_edges = gpd.GeoDataFrame(edge_geoms, crs="EPSG:4326").to_crs(3857)

    # --- Csomópontok ---
    gdf_nodes = gpd.GeoDataFrame(
        station_coords, geometry=gpd.points_from_xy(station_coords.lng, station_coords.lat), crs="EPSG:4326"
    ).to_crs(3857)

    # --- Plot két tengelyen ---
    fig, (ax_in, ax_out) = plt.subplots(1, 2, figsize=(16, 8))

    # Inflow (piros)
    gdf_in = gdf_edges[gdf_edges['direction'] == 'in']
    gdf_in.plot(ax=ax_in, color='red', linewidth=(gdf_in['weight'] / gdf_in['weight'].max()) * 5, alpha=0.7)
    for _, row in gdf_in.iterrows():
        x, y = row.geometry.interpolate(0.5, normalized=True).xy
        ax_in.text(x[0], y[0] + 30, str(int(row.weight)), fontsize=9, color='red', ha='center', va='center', fontweight='bold',
                path_effects=[path_effects.withStroke(linewidth=2,foreground='white')],)

    # Outflow (kék)
    gdf_out = gdf_edges[gdf_edges['direction'] == 'out']
    gdf_out.plot(ax=ax_out, color='blue', linewidth=(gdf_out['weight'] / gdf_out['weight'].max()) * 5, alpha=0.7)
    for _, row in gdf_out.iterrows():
        x, y = row.geometry.interpolate(0.5, normalized=True).xy
        ax_out.text(x[0], y[0] + 30, str(int(row.weight)), fontsize=9, color='blue', ha='center', va='center', fontweight='bold',
                path_effects=[path_effects.withStroke(linewidth=2,foreground='white')],)

    # Csomópontok és klaszterpont
    for ax in [ax_in, ax_out]:
        gdf_nodes.plot(ax=ax, color='black', markersize=10)
        ax.scatter(*gdf_nodes.loc[family_name, 'geometry'].coords[0], color='gold', s=80, edgecolor='black')

        # Zoom
        xmin, ymin, xmax, ymax = gdf_edges.total_bounds
        dx = (xmax - xmin) * 0.15
        dy = (ymax - ymin) * 0.15
        ax.set_xlim(xmin - dx, xmax + dx)
        ax.set_ylim(ymin - dy, ymax + dy)

        ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)

    # Címek
    ax_in.set_title(f"{family_name} – Top 10 inflow", fontsize=13, color='red')
    ax_out.set_title(f"{family_name} – Top 10 outflow", fontsize=13, color='blue')

    plt.tight_layout()
    plt.savefig(f"{family_name.replace(' ', '_').replace('–','-')}_graph_split.png", dpi=300)
    plt.close(fig)
    print(f"✅ {family_name} mentve (in/out külön ábrázolva).")


✅ Csonka János tér (2022-06-15) mentve (in/out külön ábrázolva).
✅ Zugló – Sportaréna (2023-06-28) mentve (in/out külön ábrázolva).
✅ Zugló – Vasútállomás (2023-06-28) mentve (in/out külön ábrázolva).
✅ 9. kerület – Haller–Mester (2023-09-01) mentve (in/out külön ábrázolva).
✅ Kopaszi-gát (2024-04-09) mentve (in/out külön ábrázolva).
✅ Westend – Balzac utca (2024-05-01) mentve (in/out külön ábrázolva).


In [6]:
import pandas as pd
import networkx as nx

# A teljes adat (2022–2024) beolvasása
df = pd.read_pickle("bubi_all_years.pkl")

# -------------------------------------------------------
# 1. Irányított, súlyozott él-lista előállítása
# -------------------------------------------------------

# Él súly = hány alkalommal mentek A -> B irányba
edge_weights = (
    df.groupby(['start_place_id', 'end_place_id'])
      .size()
      .reset_index(name='weight')
)

# -------------------------------------------------------
# 2. Gráf létrehozása
# -------------------------------------------------------
G = nx.DiGraph()

for _, row in edge_weights.iterrows():
    u = row['start_place_id']
    v = row['end_place_id']
    w = row['weight']
    G.add_edge(u, v, weight=w)

print("Csúcsok száma:", G.number_of_nodes())
print("Élek száma:", G.number_of_edges())

Csúcsok száma: 232
Élek száma: 45092


In [7]:
# Súlyból "költség"
for u, v, data in G.edges(data=True):
    data['cost'] = 1 / data['weight']

bet = nx.betweenness_centrality(G, weight='cost', normalized=True)

In [8]:
pr = nx.pagerank(G, weight='weight')

In [9]:
eig = nx.eigenvector_centrality_numpy(G, weight='weight')
eign = nx.eigenvector_centrality(G, max_iter=5000, weight='weight')

In [10]:
pip install python-louvain

Note: you may need to restart the kernel to use updated packages.


In [11]:
import community as community_louvain

# Louvain csak súlyozott, irányítatlan gráfon működik stabilan
# → szimmetrizáljuk a gráfot
UG = G.to_undirected()

partition = community_louvain.best_partition(
    UG, weight='weight'
)

# partition: {csúcs → közösség_azonosító}

In [12]:
results = pd.DataFrame({
    'station_id': list(G.nodes()),
    'betweenness': [bet[n] for n in G.nodes()],
    'pagerank': [pr[n] for n in G.nodes()],
    'eigenvector': [eig[n] for n in G.nodes()],
    'community': [partition.get(n, -1) for n in G.nodes()],
})

results.to_csv("bubi_network_centralities.csv", index=False)

In [16]:
import networkx as nx
import pandas as pd
import numpy as np

# --- Betöltés (állítsd be a saját hálózatodra!) ---
# G = nx.read_graphml("halozat.graphml")

# Példa lista a kiválasztott állomásokra:
kivalasztott_allomasok = [
    "0620-Westend - Balzac utca",
    "0922-Haller utca – Mester utca",
    "1128-Csonka János tér",
    "1137-Kopaszi-gát",
    "1408-Zugló vasútállomás",
    "1410-Papp László Budapest Sportaréna"
]

# --- 1) Degree asymmetry (in-degree vs out-degree arány) ---
def degree_asymmetry(G, node):
    indeg = G.in_degree(node)
    outdeg = G.out_degree(node)
    if indeg + outdeg == 0:
        return 0
    return (outdeg - indeg) / (outdeg + indeg)

deg_asym = {n: degree_asymmetry(G, n) for n in G.nodes()}

# --- 2) Local clustering coefficient ---
# Irányított gráf esetén ez unidirekcionális clustering
clustering = nx.clustering(G.to_undirected())

# --- 3) Ego-betweenness centrality ---
ego_bet = {}
for n in G.nodes():
    ego = nx.ego_graph(G, n, undirected=True)
    ego_bet[n] = nx.betweenness_centrality(ego).get(n, 0)

# --- 4) Adatok összefoglalása DataFrame-be ---
adat = []
for node in kivalasztott_allomasok:
    adat.append({
        "station": node,
        "degree_asymmetry": deg_asym.get(node, np.nan),
        "local_clustering": clustering.get(node, np.nan),
        "ego_betweenness": ego_bet.get(node, np.nan)
    })

df = pd.DataFrame(adat)

# --- 5) CSV mentése ---
df.to_csv("kivalasztott_allomasok_mutatoi.csv", index=False)

print(df)
print("\nCSV elkészült: kivalasztott_allomasok_mutatoi.csv")


                                station  degree_asymmetry  local_clustering  \
0            0620-Westend - Balzac utca         -0.015480          0.928944   
1        0922-Haller utca – Mester utca          0.006250          0.949052   
2                 1128-Csonka János tér          0.011628          0.919151   
3                      1137-Kopaszi-gát          0.011834          0.922973   
4               1408-Zugló vasútállomás          0.000000          0.919171   
5  1410-Papp László Budapest Sportaréna         -0.016949          0.927783   

   ego_betweenness  
0         0.000754  
1         0.000616  
2         0.000896  
3         0.000854  
4         0.001053  
5         0.000865  

CSV elkészült: kivalasztott_allomasok_mutatoi.csv
